<a href="https://colab.research.google.com/github/Shorovpaul/Data_Testing/blob/main/CFD02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install shap

In [ ]:
pip install imblearn

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

import xgboost as xgb

from imblearn.over_sampling import SMOTE

import shap


In [ ]:
data = pd.read_csv("creditcard.csv")

X = data.drop("Class", axis=1)
y = data["Class"]
nan_indices = y[y.isna()].index
X = X.drop(nan_indices)
y = y.drop(nan_indices)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [ ]:
smote = SMOTE(random_state=42)

X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

In [ ]:
scaler = StandardScaler()

X_train_bal = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

In [ ]:
models = {

    "LR": LogisticRegression(max_iter=3000),

    "RF": RandomForestClassifier(
        n_estimators=200,
        n_jobs=-1,
        random_state=42
    ),

    "XGB": xgb.XGBClassifier(
        tree_method="hist",
        eval_metric="logloss",
        n_jobs=-1
    )
}

trained_models = {}

for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)
    trained_models[name] = model

In [ ]:
model_scores = {}

for name, model in trained_models.items():

    probs = model.predict_proba(X_test_scaled)[:,1]

    score = average_precision_score(y_test, probs)

    model_scores[name] = score

print(model_scores)

{'LR': np.float64(0.4891259657120931), 'RF': np.float64(0.8310286329083956), 'XGB': np.float64(0.8459280463331066)}


In [ ]:
X_sample, _, y_sample, _ = train_test_split(
    X_train_bal,
    y_train_bal,
    train_size=2000,
    stratify=y_train_bal,
    random_state=42
)

In [ ]:
explanations = {}

for name, model in trained_models.items():

    explainer = shap.Explainer(model, X_sample)

    shap_values = explainer(X_sample)

    importance = np.abs(shap_values.values).mean(axis=0)

    explanations[name] = importance

100%|===================| 3986/4000 [03:28<00:00]       

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=3)

stability_scores = {}

for name, model in trained_models.items():

    fold_importance = []

    for train_idx, val_idx in kf.split(X_sample):

        model.fit(X_sample[train_idx], y_sample.iloc[train_idx])

        explainer = shap.Explainer(model, X_sample[val_idx])

        shap_values = explainer(X_sample[val_idx])

        importance = np.abs(shap_values.values).mean(axis=0)

        fold_importance.append(importance)

    fold_importance = np.array(fold_importance)

    stability = 1 / (1 + fold_importance.var())

    stability_scores[name] = stability

print(stability_scores)


100%|===================| 1330/1332 [00:21<00:00]       

{'LR': np.float64(0.6943944679442136), 'RF': np.float64(0.9992068035644386), 'XGB': np.float64(0.7694085324510594)}


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

diversity_scores = {}

# Ensure all explanation arrays are 1D (num_features,)
processed_explanations = {}
for name, importance_array in explanations.items():
    if importance_array.ndim == 2:
        # If it's (num_features, num_classes), average across classes to get (num_features,)
        processed_explanations[name] = importance_array.mean(axis=1)
    else:
        processed_explanations[name] = importance_array

names = list(processed_explanations.keys())

for i in range(len(names)):
    for j in range(i+1, len(names)):

        sim = cosine_similarity(
            processed_explanations[names[i]].reshape(1,-1),
            processed_explanations[names[j]].reshape(1,-1)
        )[0][0]

        diversity_scores[(names[i],names[j])] = 1 - sim

In [ ]:

c= {}

for name in trained_models:

    performance = model_scores[name]

    stability = stability_scores[name]

    diversity = np.mean([
        v for k,v in diversity_scores.items() if name in k
    ])

    score = (
        0.5 * performance +
        0.3 * stability +
        0.2 * diversity
    )

    final_scores[name] = score

print(final_scores)

{'LR': np.float64(0.486720317427732), 'RF': np.float64(0.7359560604791076), 'XGB': np.float64(0.6803902108894442)}


In [ ]:
print("Explanation: The `final_scores` are lower than the `model_scores` because they incorporate two additional factors: stability and diversity of the model's explanations (SHAP values), weighted alongside the initial performance (AUPRC). While `model_scores` only reflect performance, `stability` measures how consistent the feature importances are, and `diversity` measures how different the explanations are across models. Since both `stability_scores` and especially `diversity_scores` are generally lower than the initial AUPRC performance scores, their inclusion in the weighted average (with weights 0.3 for stability and 0.2 for diversity) pulls down the overall `final_score`.")

top_models = sorted(final_scores, key=final_scores.get, reverse=True)[:2]

print(top_models)

['RF', 'XGB']


In [ ]:

estimators = [(name, trained_models[name]) for name in top_models]

meta_model = LogisticRegression()

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_model
)

stack.fit(X_train_bal, y_train_bal)

StackingClassifier(estimators=[('RF',
                                RandomForestClassifier(n_estimators=200,
                                                       n_jobs=-1,
                                                       random_state=42)),
                               ('XGB',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='logloss',
                                              feature_types=None,
                                              feature_...
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_rate=None, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None,
                                              max_depth=None, max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=None, n_jobs=-1,
                                              num_parallel_tree=None, ...))],
                   final_estimator=LogisticRegression())

In [ ]:


probs = stack.predict_proba(X_test_scaled)[:,1]

auprc = average_precision_score(y_test, probs)

print("Stacked AUPRC:", auprc)

Stacked AUPRC: 0.8430062708662567
